Anime Recommendation System

In [9]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("CooperUnion/anime-recommendations-database")

print("Path to dataset files:", path)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: /Users/julieannsalangsang/.cache/kagglehub/datasets/CooperUnion/anime-recommendations-database/versions/1


PHASE 1: Content-Based Recommender

In [10]:
import pandas as pd
df = pd.read_csv(f"{path}/anime.csv")

print(df.head())

print("\nDataset Information:" )
print(df.info())

print("\nMissing Values:")
print(df.isnull().sum())

print(df.describe())

print(df["episodes"].unique()[:30]) 

   anime_id                              name  \
0     32281                    Kimi no Na wa.   
1      5114  Fullmetal Alchemist: Brotherhood   
2     28977                          Gintama°   
3      9253                       Steins;Gate   
4      9969                     Gintama&#039;   

                                               genre   type episodes  rating  \
0               Drama, Romance, School, Supernatural  Movie        1    9.37   
1  Action, Adventure, Drama, Fantasy, Magic, Mili...     TV       64    9.26   
2  Action, Comedy, Historical, Parody, Samurai, S...     TV       51    9.25   
3                                   Sci-Fi, Thriller     TV       24    9.17   
4  Action, Comedy, Historical, Parody, Samurai, S...     TV       51    9.16   

   members  
0   200630  
1   793665  
2   114262  
3   673572  
4   151266  

Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Cou

Data Cleaning

In [11]:
df["episodes"] = pd.to_numeric(
    df["episodes"],
    errors="coerce"
)

print(df.isnull().sum())

df = df.dropna(
    subset=["name", "genre", "type", "rating"]
)

df = df.reset_index(drop=True)

print("Cleaned Dataset Shape:")
print(df.shape)

print("\nMissing Values")
print(df.isnull().sum())


anime_id      0
name          0
genre        62
type         25
episodes    340
rating      230
members       0
dtype: int64
Cleaned Dataset Shape:
(12017, 7)

Missing Values
anime_id      0
name          0
genre         0
type          0
episodes    187
rating        0
members       0
dtype: int64


Prepare the features. Comparing the anime based on there genre and type

genre + type --> similarity feature --> find anime with similar content

In [16]:
recommend_df = df[[
    "anime_id", 
    "name",
    "genre",
    "type",
    "rating",
    "members"
]].copy()

print(recommend_df.head())
print(recommend_df.info())

   anime_id                              name  \
0     32281                    Kimi no Na wa.   
1      5114  Fullmetal Alchemist: Brotherhood   
2     28977                          Gintama°   
3      9253                       Steins;Gate   
4      9969                     Gintama&#039;   

                                               genre   type  rating  members  
0               Drama, Romance, School, Supernatural  Movie    9.37   200630  
1  Action, Adventure, Drama, Fantasy, Magic, Mili...     TV    9.26   793665  
2  Action, Comedy, Historical, Parody, Samurai, S...     TV    9.25   114262  
3                                   Sci-Fi, Thriller     TV    9.17   673572  
4  Action, Comedy, Historical, Parody, Samurai, S...     TV    9.16   151266  
<class 'pandas.DataFrame'>
RangeIndex: 12017 entries, 0 to 12016
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12017 non-null  int64  
 1   name      12

In [17]:
# combined feature column
recommend_df["features"] = (
    recommend_df["genre"]+ " " + 
    recommend_df["type"]
)

print(
    recommend_df[
        ["features", "genre", "type", "features"]
    ].head()
)

                                            features  \
0         Drama, Romance, School, Supernatural Movie   
1  Action, Adventure, Drama, Fantasy, Magic, Mili...   
2  Action, Comedy, Historical, Parody, Samurai, S...   
3                                Sci-Fi, Thriller TV   
4  Action, Comedy, Historical, Parody, Samurai, S...   

                                               genre   type  \
0               Drama, Romance, School, Supernatural  Movie   
1  Action, Adventure, Drama, Fantasy, Magic, Mili...     TV   
2  Action, Comedy, Historical, Parody, Samurai, S...     TV   
3                                   Sci-Fi, Thriller     TV   
4  Action, Comedy, Historical, Parody, Samurai, S...     TV   

                                            features  
0         Drama, Romance, School, Supernatural Movie  
1  Action, Adventure, Drama, Fantasy, Magic, Mili...  
2  Action, Comedy, Historical, Parody, Samurai, S...  
3                                Sci-Fi, Thriller TV  
4  Action

Convert the text features into numbers

In [21]:
# converts each anime into numerical vector based on the words in the features column 

from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()

feature_matrix = vectorizer.fit_transform(
    recommend_df["features"]
)

print(feature_matrix.shape)

# check all the generated features in the feature names
print(vectorizer.get_feature_names_out())

(12017, 52)
['action' 'adventure' 'ai' 'arts' 'cars' 'comedy' 'dementia' 'demons'
 'drama' 'ecchi' 'fantasy' 'fi' 'game' 'harem' 'hentai' 'historical'
 'horror' 'josei' 'kids' 'life' 'magic' 'martial' 'mecha' 'military'
 'movie' 'music' 'mystery' 'of' 'ona' 'ova' 'parody' 'police' 'power'
 'psychological' 'romance' 'samurai' 'school' 'sci' 'seinen' 'shoujo'
 'shounen' 'slice' 'space' 'special' 'sports' 'super' 'supernatural'
 'thriller' 'tv' 'vampire' 'yaoi' 'yuri']


Calculate Cosine Similarity between all the anime

In [22]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(feature_matrix)

print(similarity_matrix.shape)

(12017, 12017)


Build the recommendation function

In [57]:
def recommend_anime(anime_name, n=5):
    # find the anime
    matches = recommend_df[
        recommend_df["name"].str.lower() == anime_name.lower()
    ]

    if matches.empty:
        return "Anime not found."

    anime_index = matches.index[0]

    # get similarity scores
    similarity_scores = list(
        enumerate(similarity_matrix[anime_index])
    )

    # remove the selected anime itself
    similarity_scores = [
        (index, score)
        for index, score in similarity_scores
        if index != anime_index
    ]

    # sort from highest to lowest similarity
    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    # get top recommendations
    top_matches = similarity_scores[:n]

    recommended_indexes = [
        index for index, score in top_matches
    ]

    recommendations = recommend_df.loc[
        recommended_indexes,
        ["name", "genre", "type", "rating", "members"]
    ].copy()

    # add similarity score
    recommendations["similarity_score"] = [
        score for index, score in top_matches
    ]

        # sort by similarity first, then rating
    recommendations = recommendations.sort_values(
        by=["similarity_score", "rating"],
        ascending=[False, False]
    )

    return recommendations

In [61]:
# to test

#recommend_anime("Naruto",)

# ask more recommendations
recommend_anime("Naruto", n=10)

,name,genre,type,rating,members,similarity_score
615,Naruto: Shippuuden,"Action, Comedy, Martial Arts, Shounen, Super P...",TV,7.94,533578,1.000000
206,Dragon Ball Z,"Action, Adventure, Comedy, Fantasy, Martial Ar...",TV,8.32,375662,0.894427
515,Dragon Ball Kai (2014),"Action, Adventure, Comedy, Fantasy, Martial Ar...",TV,8.01,42666,0.894427
588,Dragon Ball Kai,"Action, Adventure, Comedy, Fantasy, Martial Ar...",TV,7.95,116832,0.894427
1209,Medaka Box Abnormal,"Action, Comedy, Ecchi, Martial Arts, School, S...",TV,7.63,66972,0.894427
1930,Dragon Ball Super,"Action, Adventure, Comedy, Fantasy, Martial Ar...",TV,7.40,111443,0.894427
2615,Medaka Box,"Action, Comedy, Ecchi, Martial Arts, School, S...",TV,7.21,110042,0.894427
3037,Tenjou Tenge,"Action, Comedy, Ecchi, Martial Arts, School, S...",TV,7.10,103449,0.894427
486,Boruto: Naruto the Movie,"Action, Comedy, Martial Arts, Shounen, Super P...",Movie,8.03,74690,0.875000
1103,Boruto: Naruto the Movie - Naruto ga Hokage ni...,"Action, Comedy, Martial Arts, Shounen, Super P...",Special,7.68,16868,0.875000


PHASE 2: Collaborative Filterin

Load and Inspect the dataset

In [63]:
rating_df = pd.read_csv(f"{path}/rating.csv")

print(rating_df.head())
print(rating_df.shape)
print(rating_df.info())

print(rating_df.isnull().sum())

print(rating_df["rating"].value_counts().sort_index())

   user_id  anime_id  rating
0        1        20      -1
1        1        24      -1
2        1        79      -1
3        1       226      -1
4        1       241      -1
(7813737, 3)
<class 'pandas.DataFrame'>
RangeIndex: 7813737 entries, 0 to 7813736
Data columns (total 3 columns):
 #   Column    Dtype
---  ------    -----
 0   user_id   int64
 1   anime_id  int64
 2   rating    int64
dtypes: int64(3)
memory usage: 178.8 MB
None
user_id     0
anime_id    0
rating      0
dtype: int64
rating
-1     1476496
 1       16649
 2       23150
 3       41453
 4      104291
 5      282806
 6      637775
 7     1375287
 8     1646019
 9     1254096
 10     955715
Name: count, dtype: int64


Clean the data

In [ ]:
rating_clean = rating_df[
    rating_df["rating"] != -1
].copy()

print(rating_clean.shape)

print(
    rating_clean["rating"]
    .value_counts()
    .sort_index()
)

(6337241, 3)
rating
1       16649
2       23150
3       41453
4      104291
5      282806
6      637775
7     1375287
8     1646019
9     1254096
10     955715
Name: count, dtype: int64
user_id     0
anime_id    0
rating      0
dtype: int64


In [ ]:
# missing values
print(rating_clean.isnull().sum())

user_id     0
anime_id    0
rating      0
dtype: int64


In [67]:
print("Unique_users: ", rating_clean["user_id"].nunique())
print("Unique_anime: ", rating_clean["anime_id"].nunique())

Unique_users:  69600
Unique_anime:  9927
